# Main results

The unit-level model with ethnicity-by-year fixed effects and the within-pair regression are the same estimator, so their coefficients have to match.

**Expected**

| specification | coefficient |
|---|---|
| unit and year FE | -0.2163 |
| ethnicity-by-year FE | -0.2096 |
| within-pair gap | -0.2096 |
| within-pair, year FE | -0.1656 |
| plus common trend | -0.0703 |
| plus group trends | -0.0424 |

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt

import config
import estimation

In [ ]:
panel_df = pd.read_csv(config.WORK / 'panel_narrow.csv')
pair = pd.read_csv(config.WORK / 'pair_panel.csv')
specs = estimation.main_specifications(panel_df, pair)
specs.round(4)

In [ ]:
# the two identical specifications must agree
a = specs.loc[specs['spec'] == 'ethnicity-by-year FE', 'coef'].iloc[0]
b = specs.loc[specs['spec'] == 'within-pair gap', 'coef'].iloc[0]
print(f'difference: {abs(a - b):.2e}')
assert abs(a - b) < 1e-8

In [ ]:
estimation.cohort_effects(pair).round(4)

In [ ]:
coefs, model, terms = estimation.event_study(pair)
print(estimation.pretrend_test(model, terms))
coefs.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(-0.5, color=config.ACCENT, lw=1.2, ls=':', label='SAP entry')
ax.fill_between(coefs['rel_year'], coefs['lo'], coefs['hi'],
                alpha=0.2, color=config.NAVY, label='95% CI')
ax.plot(coefs['rel_year'], coefs['coef'], 'o-', color=config.NAVY, lw=2)
ax.set_xlabel('Years relative to SAP entry')
ax.set_ylabel('Within-pair gap in ln(NTL)')
ax.legend()
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
fig.savefig(config.OUT / 'event_study.png', dpi=config.DPI, bbox_inches='tight')